# Notebook 11: Outcome Merge and Modelling

Merges early-window student features with course outcomes, builds the final and modelling datasets, and runs Leave-One-Year-Out cross-validation for four baseline models (Logistic Regression, Random Forest, Gradient Boosting, SVM).


In [1]:
# ══════════════════════════════════════
# STEP 1: Import required libraries
# ══════════════════════════════════════
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import os

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
RESULTS_DIR = "Results"
os.makedirs(RESULTS_DIR, exist_ok=True)


In [2]:
# ══════════════════════════════════════
# STEP 2: Load student feature dataset
# ══════════════════════════════════════
file_path2 = "student_features_early3w.xlsx"
student_features = pd.read_excel(file_path2)

print("Student feature dataset shape:", student_features.shape)
print("\nFeature dataset columns:")
print(student_features.columns.tolist())

Student feature dataset shape: (1154, 30)

Feature dataset columns:
['id', 'year', 'num_atmp_l1', 'num_atmp_l2', 'num_atmp_l3', 'num_atmp_l4', 'num_atmp_l5', 'num_atmp_l6', 'num_atmp_l7', 'num_atmp_l8', 'num_atmp_l11', 'wk_comp_l1', 'wk_comp_l2', 'wk_comp_l3', 'wk_comp_l4', 'wk_comp_l5', 'wk_comp_l6', 'wk_comp_l7', 'wk_comp_l8', 'total_attempts_3w', 'levels_completed_3w', 'efficiency_3w', 'gap1to2_3w', 'gap2to3_3w', 'gap3to4_3w', 'gap4to5_3w', 'gap5to6_3w', 'gap6to7_3w', 'gap7to8_3w', 'total_inactivity_3w']


In [3]:
# ══════════════════════════════════════
# STEP 3: Load outcome data from mastery_data.xlsx
# ══════════════════════════════════════
file_path3 = "mastery_data.xlsx"
excel_file = pd.ExcelFile(file_path3)
print(excel_file.sheet_names)

['Info Attempts', '2014 Assessment', '2015 Assessment', '2016 Assessment', '2017 Assessment', '2018 Assessment', 'All Attempts']


In [4]:
outcome_file = file_path3

sheet_map = {
    2014: "2014 Assessment",
    2015: "2015 Assessment",
    2016: "2016 Assessment",
    2017: "2017 Assessment",
    2018: "2018 Assessment"
}

outcome_dfs = []

for yr, sheet_name in sheet_map.items():
    temp = pd.read_excel(outcome_file, sheet_name=sheet_name)
    temp["year"] = yr
    print(f"Loaded {sheet_name}: {temp.shape[0]} records")
    outcome_dfs.append(temp)

outcomes = pd.concat(outcome_dfs, ignore_index=True)


print(f"\n Total outcome records loaded:{outcomes.shape[0]}")
print(f" Columns:{list(outcomes.columns)}")
print(f"\n First few rows:")
print(outcomes.head())

Loaded 2014 Assessment: 242 records
Loaded 2015 Assessment: 243 records


Loaded 2016 Assessment: 235 records


Loaded 2017 Assessment: 232 records


Loaded 2018 Assessment: 202 records

 Total outcome records loaded:1154
 Columns:['new ID', 'year', 'progression /35', 'practical 1 /15', 'practical 2 /50 (exam)', 'total (sum)', 'final grade', 'progression /60', 'practical 1 /20', 'practical 2 /20', 'progression /60 (late not applied)']

 First few rows:
     new ID  year  progression /35  practical 1 /15  practical 2 /50 (exam)  \
0  PXYW0522  2014             20.0              0.0                    17.0   
1  VUYC3532  2014             35.0              5.0                    41.0   
2  FLCD9260  2014             30.0              0.0                     0.0   
3  YMEY5265  2014             35.0             15.0                    45.0   
4  MNZB8487  2014             35.0             10.0                    32.0   

   total (sum) final grade  progression /60  practical 1 /20  practical 2 /20  \
0           37           E              NaN              NaN              NaN   
1           81          A-              NaN             

In [5]:
# ══════════════════════════════════════
# STEP 4: Keep only needed columns
# ══════════════════════════════════════
# We only need: new ID, year, final grade
outcomes = outcomes[["new ID", "year", "final grade"]].copy()

# Rename new ID to id so it matches student_features
outcomes = outcomes.rename(columns={"new ID": "id"})


In [6]:
# ══════════════════════════════════════
# STEP 5: Clean id and year values
# ══════════════════════════════════════
student_features["id"] = student_features["id"].astype(str).str.strip()
outcomes["id"] = outcomes["id"].astype(str).str.strip()

student_features["year"] = pd.to_numeric(student_features["year"], errors="coerce")
outcomes["year"] = pd.to_numeric(outcomes["year"], errors="coerce")


In [7]:
# Check for missing values in year column

print(student_features["year"].isna().sum())
print(outcomes["year"].isna().sum())

0
0


In [8]:
# ══════════════════════════════════════
# STEP 6: Create binary outcome variable
# ══════════════════════════════════════
# Pass = 1 and Fail = 0
# D and E are fail
def map_grade_to_outcome(grade):
    if pd.isna(grade):
        return np.nan

    grade = str(grade).strip().upper()

    # D and E grades = fail
    if grade.startswith("D") or grade.startswith("E"):
        return 0

    # A, B, C grades = pass
    elif grade.startswith("A") or grade.startswith("B") or grade.startswith("C"):
        return 1

    # Any unknown code becomes missing
    else:
        return np.nan

outcomes["outcome"] = outcomes["final grade"].apply(map_grade_to_outcome)

print("\nFinal grade distribution:")
print(outcomes["final grade"].value_counts(dropna=False).sort_index())

print("\nBinary outcome distribution:")
print(outcomes["outcome"].value_counts(dropna=False))



Final grade distribution:
final grade
A      94
A+    177
A-     71
B      77
B+     86
B-     76
C     107
C+     84
C-    115
D      63
E     204
Name: count, dtype: int64

Binary outcome distribution:
outcome
1    887
0    267
Name: count, dtype: int64


Final letter grades were converted into a binary outcome variable, where grades A to C were coded as pass (1) and grades D and E were coded as fail (0). This produced 887 pass cases and 267 fail cases across 1154 student-cohort records. As expected, this dataset is highly imbalanced with 76.9% Pass and 23.1% Fail, which further justifies the need for a robust classifier like Random Forest and evaluation metrics like Recall, F1-Score, and AUC. 

In [9]:
# ══════════════════════════════════════
# STEP 7: Check student-cohort grade records against mastery feature records
# ══════════════════════════════════════
# This checks whether each (id, year) in the grade file has a matching (id, year) in the feature file.
grade_keys = outcomes[["id", "year"]].drop_duplicates()
feature_keys = student_features[["id", "year"]].drop_duplicates()

grade_vs_feature = grade_keys.merge(
    feature_keys,
    on=["id", "year"],
    how="left",
    indicator=True
)

n_grade_records = len(grade_keys)
n_feature_records = len(feature_keys)
n_matched = (grade_vs_feature["_merge"] == "both").sum()
n_grade_no_feature = (grade_vs_feature["_merge"] == "left_only").sum()

print("\n" + "="*60)
print("GRADE RECORDS VS MASTERY FEATURE RECORDS")
print("="*60)
print("Number of student-cohort grade records:", n_grade_records)
print("Number of student-cohort feature records:", n_feature_records)
print("Number with both grade and feature record:", n_matched)
print("Number with grade but NO feature record:", n_grade_no_feature)

if n_grade_no_feature > 0:
    print("\nStudent-cohort instances with grade but no feature record:")
    print(grade_vs_feature[grade_vs_feature["_merge"] == "left_only"].head(20))
else:
    print("\nAll student-cohort grade records have a matching feature record.")



GRADE RECORDS VS MASTERY FEATURE RECORDS
Number of student-cohort grade records: 1154
Number of student-cohort feature records: 1154
Number with both grade and feature record: 1154
Number with grade but NO feature record: 0

All student-cohort grade records have a matching feature record.


A consistency check was performed before merging the outcome and feature datasets. Unique student-cohort identifiers (id, year) were compared across both datasets. All 1154 outcome records had matching feature records, indicating complete alignment between the two data sources and no loss of student-cohort cases at the merge stage. 

In [10]:
# ══════════════════════════════════════
# STEP 8: Check students with final grade
# but zero mastery attempts in Weeks 1-3
# ══════════════════════════════════════
if "total_attempts_3w" in student_features.columns:
    zero_attempt_keys = student_features.loc[
        student_features["total_attempts_3w"] == 0, ["id", "year"]
    ].drop_duplicates()

    grade_but_zero_attempts = outcomes.merge(
        zero_attempt_keys,
        on=["id", "year"],
        how="inner"
    )

    print("\n" + "="*60)
    print("STUDENTS WITH FINAL GRADE BUT ZERO EARLY MASTERY ATTEMPTS")
    print("="*60)
    print(
        "Number of student-cohort instances with final grade but 0 attempts in Weeks 1-3:",
        grade_but_zero_attempts[["id", "year"]].drop_duplicates().shape[0]
    )

    if len(grade_but_zero_attempts) > 0:
        print("\nExamples:")
        print(
            grade_but_zero_attempts[["id", "year", "final grade", "outcome"]]
            .drop_duplicates()
            .head(20)
        )
else:
    print("\nWARNING: Column 'total_attempts_3w' was not found in student_features.")


STUDENTS WITH FINAL GRADE BUT ZERO EARLY MASTERY ATTEMPTS
Number of student-cohort instances with final grade but 0 attempts in Weeks 1-3: 21

Examples:
          id  year final grade  outcome
0   BYRN8962  2014           E        0
1   RVMF4552  2015          C-        1
2   DBVM4419  2015           E        0
3   CEDR5183  2015          C-        1
4   SLLP1898  2016           C        1
5   TXRA4860  2016          C-        1
6   DYVF2928  2016          C-        1
7   OBWM7893  2016           E        0
8   BEDE6852  2016           E        0
9   HOQL8508  2016           E        0
10  ZCSK9917  2016           E        0
11  KIAM7733  2016           E        0
12  SVNP1242  2016           E        0
13  GXLL1492  2016           E        0
14  LRLB2557  2016           E        0
15  CEZB8130  2017          B+        1
16  BDUO0575  2017           C        1
17  HGRA1893  2017           D        0
18  XEQB3620  2017           E        0
19  GYIY5442  2017           E        0


In [11]:
# ══════════════════════════════════════
# STEP 9a: Merge features with outcomes
# ══════════════════════════════════════
final_dataset = student_features.merge(
    outcomes[["id", "year", "final grade", "outcome"]],
    on=["id", "year"],
    how="inner"
)

# Remove rows with missing outcome
final_dataset = final_dataset.dropna(subset=["outcome"]).copy()
final_dataset["outcome"] = final_dataset["outcome"].astype(int)

print("\nMerged final dataset shape:", final_dataset.shape)

print("\nOutcome distribution in final dataset:")
print(final_dataset["outcome"].value_counts())

print("\nNumber of unique students:", final_dataset["id"].nunique())
print("Number of unique student-cohort records:", final_dataset[["id", "year"]].drop_duplicates().shape[0])


# ══════════════════════════════════════
# STEP 9b: Create modelling dataset
# ══════════════════════════════════════
modeling_data = final_dataset.drop(columns=["id", "final grade"]).copy()

print("\nModeling dataset shape:", modeling_data.shape)

print("\nOutcome distribution in modeling dataset:")
print(modeling_data["outcome"].value_counts())

print("\nColumns in modeling dataset:")
print(modeling_data.columns.tolist())


# ══════════════════════════════════════
# STEP 9c: Export both datasets
# ══════════════════════════════════════
final_dataset.to_excel("final_dataset.xlsx", index=False)
final_dataset.to_csv("final_dataset.csv", index=False)

modeling_data.to_excel("modeling_dataset.xlsx", index=False)
modeling_data.to_csv("modeling_dataset.csv", index=False)


Merged final dataset shape: (1154, 32)

Outcome distribution in final dataset:
outcome
1    887
0    267
Name: count, dtype: int64

Number of unique students: 1135
Number of unique student-cohort records: 1154

Modeling dataset shape: (1154, 30)

Outcome distribution in modeling dataset:
outcome
1    887
0    267
Name: count, dtype: int64

Columns in modeling dataset:
['year', 'num_atmp_l1', 'num_atmp_l2', 'num_atmp_l3', 'num_atmp_l4', 'num_atmp_l5', 'num_atmp_l6', 'num_atmp_l7', 'num_atmp_l8', 'num_atmp_l11', 'wk_comp_l1', 'wk_comp_l2', 'wk_comp_l3', 'wk_comp_l4', 'wk_comp_l5', 'wk_comp_l6', 'wk_comp_l7', 'wk_comp_l8', 'total_attempts_3w', 'levels_completed_3w', 'efficiency_3w', 'gap1to2_3w', 'gap2to3_3w', 'gap3to4_3w', 'gap4to5_3w', 'gap5to6_3w', 'gap6to7_3w', 'gap7to8_3w', 'total_inactivity_3w', 'outcome']


In [12]:
# ══════════════════════════════════════
# STEP 10: Check repeat students
# ══════════════════════════════════════
# Repeat students are checked in final_dataset because id is not kept in modeling_data.
repeat_counts = final_dataset["id"].value_counts()
repeated_students = repeat_counts[repeat_counts > 1]

print("\nNumber of students appearing in multiple cohorts:", len(repeated_students))
print("Number of repeated student-cohort rows:", repeated_students.sum())

if len(repeated_students) > 0:
    print("\nRepeated student IDs:")
    print(repeated_students.head)


Number of students appearing in multiple cohorts: 18
Number of repeated student-cohort rows: 37

Repeated student IDs:
<bound method NDFrame.head of id
PWPO9424    3
BGMN5662    2
TXRA4860    2
UQEB3636    2
PEIQ9634    2
GQTE6968    2
DBVM4419    2
WDME3259    2
FHAR8432    2
GYZL8284    2
MHDM5723    2
JVAL5945    2
ANND1245    2
KVAW4798    2
QWNT7178    2
ETUL8523    2
FKJJ0623    2
RCQU7334    2
Name: count, dtype: int64>


Eighteen students appeared in more than one cohort year. These students contributed 37 student-cohort records in total, with 17 students appearing twice and 1 student appearing three times. Repeat enrolments were retained as separate student-cohort instances defined by (id, year). The records of those who repeat the course will be treated as separate during the model stage. 

In [13]:
# ══════════════════════════════════════
# STEP 11: Define X and y
# ══════════════════════════════════════
# modeling_data keeps year for cohort-based splitting,
# but year is NOT used as a predictor.
feature_cols = [col for col in modeling_data.columns if col not in ["year", "outcome"]]

X = modeling_data[feature_cols]
y = modeling_data["outcome"]

print("\nNumber of predictor variables:", len(feature_cols))
print("Predictor columns:")
print(feature_cols)


Number of predictor variables: 28
Predictor columns:
['num_atmp_l1', 'num_atmp_l2', 'num_atmp_l3', 'num_atmp_l4', 'num_atmp_l5', 'num_atmp_l6', 'num_atmp_l7', 'num_atmp_l8', 'num_atmp_l11', 'wk_comp_l1', 'wk_comp_l2', 'wk_comp_l3', 'wk_comp_l4', 'wk_comp_l5', 'wk_comp_l6', 'wk_comp_l7', 'wk_comp_l8', 'total_attempts_3w', 'levels_completed_3w', 'efficiency_3w', 'gap1to2_3w', 'gap2to3_3w', 'gap3to4_3w', 'gap4to5_3w', 'gap5to6_3w', 'gap6to7_3w', 'gap7to8_3w', 'total_inactivity_3w']


In [14]:
# ═════════════════════════════════════════════════
# STEP 12: Logistic Regression Leave-One-Year-Out CV
# ═════════════════════════════════════════════════
# Test one cohort year at a time.
# Repeat students are kept as separate student-cohort records.
results = []

test_years = sorted(modeling_data["year"].dropna().unique())

for test_year in test_years:
    print("\n" + "="*60)
    print("TEST YEAR:", int(test_year))
    print("="*60)

    # Test set = this year
    test_mask = modeling_data["year"] == test_year

    # Training set = all other years
    X_train = X.loc[~test_mask].copy()
    X_test = X.loc[test_mask].copy()
    y_train = y.loc[~test_mask].copy()
    y_test = y.loc[test_mask].copy()

    print("Training size:", X_train.shape[0])
    print("Test size:", X_test.shape[0])

    print("\nTraining outcome distribution:")
    print(y_train.value_counts())

    print("\nTest outcome distribution:")
    print(y_test.value_counts())

    # Scale using training data only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Logistic regression model
    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    recall_pass = recall_score(y_test, y_pred, pos_label=1)
    f1_pass = f1_score(y_test, y_pred, pos_label=1)

    recall_fail = recall_score(y_test, y_pred, pos_label=0)
    f1_fail = f1_score(y_test, y_pred, pos_label=0)

    # Confusion matrix
    # rows = actual [0,1]
    # cols = predicted [0,1]
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    results.append({
        "test_year": int(test_year),
        "train_n": len(X_train),
        "test_n": len(X_test),
        "accuracy": accuracy,
        "auc": auc,
        "recall_pass": recall_pass,
        "f1_pass": f1_pass,
        "recall_fail": recall_fail,
        "f1_fail": f1_fail,
        "actual_fail_pred_fail": tn,
        "actual_fail_pred_pass": fp,
        "actual_pass_pred_fail": fn,
        "actual_pass_pred_pass": tp
    })

    print("\nFold results:")
    print(f"Accuracy:      {accuracy:.3f}")
    print(f"AUC:           {auc:.3f}")
    print(f"Recall (Pass): {recall_pass:.3f}")
    print(f"F1 (Pass):     {f1_pass:.3f}")
    print(f"Recall (Fail): {recall_fail:.3f}")
    print(f"F1 (Fail):     {f1_fail:.3f}")

    print("\nConfusion matrix interpretation:")
    print("actual_fail_pred_fail =", tn, "-> correctly identified at-risk students")
    print("actual_fail_pred_pass =", fp, "-> missed at-risk students")
    print("actual_pass_pred_fail =", fn, "-> students incorrectly flagged as at risk")
    print("actual_pass_pred_pass =", tp, "-> correctly identified passing students")



TEST YEAR: 2014
Training size: 912
Test size: 242

Training outcome distribution:
outcome
1    691
0    221
Name: count, dtype: int64

Test outcome distribution:
outcome
1    196
0     46
Name: count, dtype: int64

Fold results:
Accuracy:      0.822
AUC:           0.771
Recall (Pass): 0.888
F1 (Pass):     0.890
Recall (Fail): 0.543
F1 (Fail):     0.538

Confusion matrix interpretation:
actual_fail_pred_fail = 25 -> correctly identified at-risk students
actual_fail_pred_pass = 21 -> missed at-risk students
actual_pass_pred_fail = 22 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 174 -> correctly identified passing students

TEST YEAR: 2015
Training size: 911
Test size: 243

Training outcome distribution:
outcome
1    703
0    208
Name: count, dtype: int64

Test outcome distribution:
outcome
1    184
0     59
Name: count, dtype: int64

Fold results:
Accuracy:      0.708
AUC:           0.748
Recall (Pass): 0.734
F1 (Pass):     0.792
Recall (Fail): 0.627
F1 (Fail):    

In [15]:
# ══════════════════════════════════════
# STEP 13: Show per-year results
# ══════════════════════════════════════
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("PER-YEAR CROSS-VALIDATION RESULTS")
print("="*60)
print(results_df.round(3))



PER-YEAR CROSS-VALIDATION RESULTS
   test_year  train_n  test_n  accuracy    auc  recall_pass  f1_pass  \
0       2014      912     242     0.822  0.771        0.888    0.890   
1       2015      911     243     0.708  0.748        0.734    0.792   
2       2016      919     235     0.740  0.811        0.707    0.801   
3       2017      922     232     0.754  0.861        0.724    0.825   
4       2018      952     202     0.807  0.826        0.946    0.878   

   recall_fail  f1_fail  actual_fail_pred_fail  actual_fail_pred_pass  \
0        0.543    0.538                     25                     21   
1        0.627    0.510                     37                     22   
2        0.836    0.626                     51                     10   
3        0.872    0.590                     41                      6   
4        0.426    0.541                     23                     31   

   actual_pass_pred_fail  actual_pass_pred_pass  
0                     22                   

# **Overall summary**

Across the five cohort years, the logistic regression model showed **moderate predictive performance overall**. Its ability to distinguish between passing and failing students was generally acceptable, with **AUC values ranging from 0.748 to 0.861**. However, its effectiveness in identifying failing students varied noticeably across cohorts. **Fail recall ranged from 0.426 to 0.872**, indicating that early mastery-test behaviour was informative, but not equally reliable in every year.

The strongest years for detecting at-risk students were **2016 and 2017**, while performance was weaker in **2014 and especially 2018**. Overall, these results suggest that early mastery-level test activity contains useful predictive signal for identifying students at risk of failure, but the model’s sensitivity to failing students is somewhat cohort-dependent.

---

## 2014

The logistic regression model performed reasonably well in 2014, with **accuracy 0.822** and **AUC 0.772**. It identified passing students effectively, with **pass recall 0.888**, but was only moderately successful at detecting failing students, with **fail recall 0.543**. The model correctly identified 25 of the 46 failing students and missed 21. This suggests that the early mastery features provided useful information, but not enough for highly reliable early identification of at-risk students in this cohort.

---

## 2015

This was one of the weaker-performing years. **Accuracy fell to 0.708** and **AUC to 0.748**, the lowest of all cohorts. Although **fail recall increased slightly to 0.627**, the model also produced many false positives, incorrectly flagging 49 passing students as at risk. This pattern suggests that early mastery behaviour was less clearly associated with final course outcome in 2015, reducing the model’s overall reliability.

---

## 2016

The 2016 cohort produced one of the stronger results for at-risk detection. The model achieved **AUC 0.811** and a high **fail recall of 0.836**, correctly identifying 51 of 61 failing students. This indicates strong sensitivity to the fail class, which is desirable for early intervention. However, the model also incorrectly flagged 51 passing students as at risk. Even so, this year shows that early mastery-level test activity can be highly informative for identifying students at risk of failure.

---

## 2017

This was the best-performing cohort overall. The model achieved the highest **AUC (0.861)** and the highest **fail recall (0.872)**. It correctly identified 41 of the 47 failing students and missed only 6. Although it also generated many false positives, the model was highly sensitive to at-risk students in this year. As a result, 2017 provides the strongest evidence of the predictive usefulness of Week 1-3 mastery-test behaviour.

---

## 2018

The 2018 cohort showed a different pattern. Overall performance appeared strong, with **accuracy 0.807**, **AUC 0.826**, and very high **pass recall 0.946**. However, **fail recall dropped to 0.426**, the lowest of all years. The model identified only 23 of 54 failing students and missed 31. This means that while the model performed well at recognising students who would pass, it was much less effective at identifying those at risk of failure in this cohort.

In [16]:
# ══════════════════════════════════════
# STEP 14: Show mean and standard deviation
# ══════════════════════════════════════
metric_cols = [
    "accuracy", "auc", "recall_pass",
    "f1_pass", "recall_fail", "f1_fail"
]

summary_df = pd.DataFrame({
    "mean": results_df[metric_cols].mean(),
    "std": results_df[metric_cols].std()
}).round(3)

print("\n" + "="*60)
print("LOGISTIC REGRESSION MEAN PERFORMANCE ACROSS ALL YEARS")
print("="*60)
print(summary_df)



LOGISTIC REGRESSION MEAN PERFORMANCE ACROSS ALL YEARS
              mean    std
accuracy     0.766  0.047
auc          0.803  0.045
recall_pass  0.800  0.109
f1_pass      0.837  0.045
recall_fail  0.661  0.191
f1_fail      0.561  0.046


In [17]:
# ══════════════════════════════════════
# STEP 15: Save cross-validation outputs
# ══════════════════════════════════════
results_df.to_csv(os.path.join(RESULTS_DIR, "logreg_loyocv_results.csv"), index=False)
summary_df.to_csv(os.path.join(RESULTS_DIR, "logreg_loyocv_summary.csv"))

print("\nSaved files:")
print("- logreg_loyocv_results.csv")
print("- logreg_loyocv_summary.csv")



Saved files:
- logreg_loyocv_results.csv
- logreg_loyocv_summary.csv


# **Logistic Regression Mean Performance**

### **Accuracy: 0.766 (std 0.047)**
On average, the model got about **77% of predictions correct**, with small variation across years.

---

### **AUC: 0.803 (std 0.045)**
The model’s ability to separate passing vs failing students was **good overall**, and fairly consistent across years.

---

### **Recall (Pass): 0.800 (std 0.109)**
The model correctly identified about **80% of passing students**.  
This was quite stable, though some years were better than others.

---

### **F1 (Pass): 0.837 (std 0.045)**
Predictions for passing students were **strong and reliable**, with good balance between precision and recall.

---

### **Recall (Fail): 0.661 (std 0.191)**
On average, the model caught about **66% of failing students**, but this varied a lot between years.  
This is the most unstable metric.

---

### **F1 (Fail): 0.561 (std 0.046)**
Performance on failing students was **moderate**, showing the model struggled more with the at-risk group than with the passing group.

---

# **Overall Summary**
The model performed well overall and consistently identified passing students, but its ability to detect failing students was more uneven and varied significantly across cohorts.

In [18]:
# ══════════════════════════════════════
# STEP 16: Random Forest Leave-One-Year-Out CV
# ══════════════════════════════════════

rf_results = []

test_years = sorted(modeling_data["year"].dropna().unique())

for test_year in test_years:
    print("\n" + "="*60)
    print("RANDOM FOREST - TEST YEAR:", int(test_year))
    print("="*60)

    # Test set = this year
    test_mask = modeling_data["year"] == test_year

    # Training set = all other years
    X_train = X.loc[~test_mask].copy()
    X_test = X.loc[test_mask].copy()
    y_train = y.loc[~test_mask].copy()
    y_test = y.loc[test_mask].copy()

    print("Training size:", X_train.shape[0])
    print("Test size:", X_test.shape[0])

    print("\nTraining outcome distribution:")
    print(y_train.value_counts())

    print("\nTest outcome distribution:")
    print(y_test.value_counts())

    # Random Forest model
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train)

    # Predict
    y_pred = rf_model.predict(X_test)
    y_prob = rf_model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    recall_pass = recall_score(y_test, y_pred, pos_label=1)
    f1_pass = f1_score(y_test, y_pred, pos_label=1)

    recall_fail = recall_score(y_test, y_pred, pos_label=0)
    f1_fail = f1_score(y_test, y_pred, pos_label=0)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    rf_results.append({
        "test_year": int(test_year),
        "train_n": len(X_train),
        "test_n": len(X_test),
        "accuracy": accuracy,
        "auc": auc,
        "recall_pass": recall_pass,
        "f1_pass": f1_pass,
        "recall_fail": recall_fail,
        "f1_fail": f1_fail,
        "actual_fail_pred_fail": tn,
        "actual_fail_pred_pass": fp,
        "actual_pass_pred_fail": fn,
        "actual_pass_pred_pass": tp
    })

    print("\nFold results:")
    print(f"Accuracy:      {accuracy:.3f}")
    print(f"AUC:           {auc:.3f}")
    print(f"Recall (Pass): {recall_pass:.3f}")
    print(f"F1 (Pass):     {f1_pass:.3f}")
    print(f"Recall (Fail): {recall_fail:.3f}")
    print(f"F1 (Fail):     {f1_fail:.3f}")

    print("\nConfusion matrix interpretation:")
    print("actual_fail_pred_fail =", tn, "-> correctly identified at-risk students")
    print("actual_fail_pred_pass =", fp, "-> missed at-risk students")
    print("actual_pass_pred_fail =", fn, "-> students incorrectly flagged as at risk")
    print("actual_pass_pred_pass =", tp, "-> correctly identified passing students")


RANDOM FOREST - TEST YEAR: 2014
Training size: 912
Test size: 242

Training outcome distribution:
outcome
1    691
0    221
Name: count, dtype: int64

Test outcome distribution:
outcome
1    196
0     46
Name: count, dtype: int64



Fold results:
Accuracy:      0.826
AUC:           0.768
Recall (Pass): 0.893
F1 (Pass):     0.893
Recall (Fail): 0.543
F1 (Fail):     0.543

Confusion matrix interpretation:
actual_fail_pred_fail = 25 -> correctly identified at-risk students
actual_fail_pred_pass = 21 -> missed at-risk students
actual_pass_pred_fail = 21 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 175 -> correctly identified passing students

RANDOM FOREST - TEST YEAR: 2015
Training size: 911
Test size: 243

Training outcome distribution:
outcome
1    703
0    208
Name: count, dtype: int64

Test outcome distribution:
outcome
1    184
0     59
Name: count, dtype: int64



Fold results:
Accuracy:      0.741
AUC:           0.738
Recall (Pass): 0.788
F1 (Pass):     0.822
Recall (Fail): 0.593
F1 (Fail):     0.526

Confusion matrix interpretation:
actual_fail_pred_fail = 35 -> correctly identified at-risk students
actual_fail_pred_pass = 24 -> missed at-risk students
actual_pass_pred_fail = 39 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 145 -> correctly identified passing students

RANDOM FOREST - TEST YEAR: 2016
Training size: 919
Test size: 235

Training outcome distribution:
outcome
1    713
0    206
Name: count, dtype: int64

Test outcome distribution:
outcome
1    174
0     61
Name: count, dtype: int64



Fold results:
Accuracy:      0.711
AUC:           0.801
Recall (Pass): 0.661
F1 (Pass):     0.772
Recall (Fail): 0.852
F1 (Fail):     0.605

Confusion matrix interpretation:
actual_fail_pred_fail = 52 -> correctly identified at-risk students
actual_fail_pred_pass = 9 -> missed at-risk students
actual_pass_pred_fail = 59 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 115 -> correctly identified passing students

RANDOM FOREST - TEST YEAR: 2017
Training size: 922
Test size: 232

Training outcome distribution:
outcome
1    702
0    220
Name: count, dtype: int64

Test outcome distribution:
outcome
1    185
0     47
Name: count, dtype: int64



Fold results:
Accuracy:      0.741
AUC:           0.805
Recall (Pass): 0.741
F1 (Pass):     0.820
Recall (Fail): 0.745
F1 (Fail):     0.538

Confusion matrix interpretation:
actual_fail_pred_fail = 35 -> correctly identified at-risk students
actual_fail_pred_pass = 12 -> missed at-risk students
actual_pass_pred_fail = 48 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 137 -> correctly identified passing students

RANDOM FOREST - TEST YEAR: 2018
Training size: 952
Test size: 202

Training outcome distribution:
outcome
1    739
0    213
Name: count, dtype: int64

Test outcome distribution:
outcome
1    148
0     54
Name: count, dtype: int64



Fold results:
Accuracy:      0.767
AUC:           0.806
Recall (Pass): 0.905
F1 (Pass):     0.851
Recall (Fail): 0.389
F1 (Fail):     0.472

Confusion matrix interpretation:
actual_fail_pred_fail = 21 -> correctly identified at-risk students
actual_fail_pred_pass = 33 -> missed at-risk students
actual_pass_pred_fail = 14 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 134 -> correctly identified passing students


In [19]:
# ══════════════════════════════════════
# STEP 17: Show Random Forest per-year results
# ══════════════════════════════════════
rf_results_df = pd.DataFrame(rf_results)

print("\n" + "="*60)
print("RANDOM FOREST PER-YEAR CROSS-VALIDATION RESULTS")
print("="*60)
print(rf_results_df.round(3))


RANDOM FOREST PER-YEAR CROSS-VALIDATION RESULTS
   test_year  train_n  test_n  accuracy    auc  recall_pass  f1_pass  \
0       2014      912     242     0.826  0.768        0.893    0.893   
1       2015      911     243     0.741  0.738        0.788    0.822   
2       2016      919     235     0.711  0.801        0.661    0.772   
3       2017      922     232     0.741  0.805        0.741    0.820   
4       2018      952     202     0.767  0.806        0.905    0.851   

   recall_fail  f1_fail  actual_fail_pred_fail  actual_fail_pred_pass  \
0        0.543    0.543                     25                     21   
1        0.593    0.526                     35                     24   
2        0.852    0.605                     52                      9   
3        0.745    0.538                     35                     12   
4        0.389    0.472                     21                     33   

   actual_pass_pred_fail  actual_pass_pred_pass  
0                     21     

# **Overall summary**

Across the five cohorts, Random Forest showed **moderate overall predictive performance**, with AUC values ranging from **0.738 to 0.806**. Its main strength was that it sometimes achieved good fail recall, especially in **2016** and to a lesser extent **2017**. However, performance for identifying failing students was inconsistent across years, dropping notably in **2018** and remaining only moderate in **2014** and **2015**.

A clear pattern is that when Random Forest achieved stronger fail recall, it often did so by flagging many passing students as at risk. This means it could be useful as an early-warning model, but it is not highly stable across cohorts and may generate substantial false positives in some years.

---

## 2014
Random Forest performed fairly well overall in 2014, with **accuracy 0.826** and **AUC 0.768**. It identified passing students well, with **pass recall 0.893**, but fail recall was only **0.543**, meaning it correctly identified 25 of 46 failing students and missed 21. This is a moderate result for at-risk detection. The model performed well overall, but its ability to detect failing students was limited.

---

## 2015
Performance in 2015 was somewhat weaker overall. **Accuracy was 0.741** and **AUC was 0.738**, the lowest AUC across the five years. The model correctly identified **59.3%** of failing students, which is moderate, and incorrectly flagged 39 passing students as at risk. This suggests that the model had some predictive value in 2015, but the separation between pass and fail cases was not especially strong in this cohort.

---

## 2016
This was one of the strongest years for identifying failing students. The model achieved **AUC 0.801** and **fail recall 0.852**, meaning it correctly identified 52 of the 61 failing students and missed only 9. That is a strong result for your research objective. However, the model also incorrectly flagged 59 passing students as at risk, which reduced overall accuracy to **0.711**. So in 2016 the model was highly sensitive to failing students, but it did so by accepting many false positives.

---

## 2017
Performance in 2017 was moderate to good. **Accuracy was 0.741** and **AUC was 0.805**, while **fail recall reached 0.745**. The model correctly identified 35 of the 47 failing students and missed 12. This is a useful level of at-risk detection, though not as strong as 2016. As in other years, there was a trade-off: 48 passing students were incorrectly flagged as at risk. Overall, this year shows that the model captured useful fail-related signal, but with moderate false-positive cost.

---

## 2018
The 2018 fold showed a weaker result for the fail class. **Accuracy was 0.767** and **AUC was 0.806**, so overall ranking performance was still reasonable, but **fail recall fell to 0.389**, the lowest of all five years. The model correctly identified only 21 of 54 failing students and missed 33. At the same time, it did very well at recognising passing students, with **pass recall 0.905**. So in 2018 the model was comparatively conservative: it was strong at predicting pass, but weak at identifying many of the students who ultimately failed.


In [20]:
# ══════════════════════════════════════
# STEP 18: Show Random Forest mean and standard deviation
# ══════════════════════════════════════
metric_cols = [
    "accuracy", "auc", "recall_pass",
    "f1_pass", "recall_fail", "f1_fail"
]

rf_summary_df = pd.DataFrame({
    "mean": rf_results_df[metric_cols].mean(),
    "std": rf_results_df[metric_cols].std()
}).round(3)

print("\n" + "="*60)
print("RANDOM FOREST MEAN PERFORMANCE ACROSS ALL YEARS")
print("="*60)
print(rf_summary_df)


RANDOM FOREST MEAN PERFORMANCE ACROSS ALL YEARS
              mean    std
accuracy     0.757  0.044
auc          0.784  0.030
recall_pass  0.798  0.103
f1_pass      0.831  0.045
recall_fail  0.625  0.180
f1_fail      0.537  0.047


# **Random Forest Mean Performance**

### **Accuracy: 0.757 (std 0.044)**
The model correctly predicted about **76% of outcomes on average**, with fairly small variation across years.

---

### **AUC: 0.784 (std 0.030)**
Its ability to separate passing vs failing students was **good but slightly weaker** than the logistic regression model.  
The low standard deviation shows the model behaved **consistently** across years.

---

### **Recall (Pass): 0.798 (std 0.103)**
The model correctly identified about **80% of passing students**, with some year-to-year variation.

---

### **F1 (Pass): 0.831 (std 0.045)**
Predictions for passing students were **strong and reliable**, similar to logistic regression.

---

### **Recall (Fail): 0.625 (std 0.180)**
On average, the model identified **62% of failing students**, but this varied a lot between cohorts.  
This is the most unstable metric, showing that at-risk detection was inconsistent.

---

### **F1 (Fail): 0.537 (std 0.047)**
Performance on failing students was **moderate**, indicating the model struggled more with the at-risk group than with the passing group.

---

# **Overall Summary**
Random Forest performed reasonably well overall and was consistently good at identifying passing students. However, its ability to detect failing students was **weaker and more variable**, suggesting that early mastery-test behaviour was informative but not equally predictive across all cohorts.


In [21]:
# ══════════════════════════════════════
# STEP 19: Save Random Forest outputs
# ══════════════════════════════════════
rf_results_df.to_csv(os.path.join(RESULTS_DIR, "random_forest_loyocv_results.csv"), index=False)
rf_summary_df.to_csv(os.path.join(RESULTS_DIR, "random_forest_loyocv_summary.csv"))

print("\nSaved files:")
print("- random_forest_loyocv_results.csv")
print("- random_forest_loyocv_summary.csv")


Saved files:
- random_forest_loyocv_results.csv
- random_forest_loyocv_summary.csv


In [22]:
# ══════════════════════════════════════
# STEP 20: Gradient Boosting Leave-One-Year-Out CV
# ══════════════════════════════════════
gb_results = []

test_years = sorted(modeling_data["year"].dropna().unique())

for test_year in test_years:
    print("\n" + "="*60)
    print("GRADIENT BOOSTING - TEST YEAR:", int(test_year))
    print("="*60)

    # Test set = this year
    test_mask = modeling_data["year"] == test_year

    # Training set = all other years
    X_train = X.loc[~test_mask].copy()
    X_test = X.loc[test_mask].copy()
    y_train = y.loc[~test_mask].copy()
    y_test = y.loc[test_mask].copy()

    print("Training size:", X_train.shape[0])
    print("Test size:", X_test.shape[0])

    print("\nTraining outcome distribution:")
    print(y_train.value_counts())

    print("\nTest outcome distribution:")
    print(y_test.value_counts())

    # Gradient Boosting model
    gb_model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    gb_model.fit(X_train, y_train)

    # Predict
    y_pred = gb_model.predict(X_test)
    y_prob = gb_model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    recall_pass = recall_score(y_test, y_pred, pos_label=1)
    f1_pass = f1_score(y_test, y_pred, pos_label=1)

    recall_fail = recall_score(y_test, y_pred, pos_label=0)
    f1_fail = f1_score(y_test, y_pred, pos_label=0)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    gb_results.append({
        "test_year": int(test_year),
        "train_n": len(X_train),
        "test_n": len(X_test),
        "accuracy": accuracy,
        "auc": auc,
        "recall_pass": recall_pass,
        "f1_pass": f1_pass,
        "recall_fail": recall_fail,
        "f1_fail": f1_fail,
        "actual_fail_pred_fail": tn,
        "actual_fail_pred_pass": fp,
        "actual_pass_pred_fail": fn,
        "actual_pass_pred_pass": tp
    })

    print("\nFold results:")
    print(f"Accuracy:      {accuracy:.3f}")
    print(f"AUC:           {auc:.3f}")
    print(f"Recall (Pass): {recall_pass:.3f}")
    print(f"F1 (Pass):     {f1_pass:.3f}")
    print(f"Recall (Fail): {recall_fail:.3f}")
    print(f"F1 (Fail):     {f1_fail:.3f}")

    print("\nConfusion matrix interpretation:")
    print("actual_fail_pred_fail =", tn, "-> correctly identified at-risk students")
    print("actual_fail_pred_pass =", fp, "-> missed at-risk students")
    print("actual_pass_pred_fail =", fn, "-> students incorrectly flagged as at risk")
    print("actual_pass_pred_pass =", tp, "-> correctly identified passing students")


GRADIENT BOOSTING - TEST YEAR: 2014
Training size: 912
Test size: 242

Training outcome distribution:
outcome
1    691
0    221
Name: count, dtype: int64

Test outcome distribution:
outcome
1    196
0     46
Name: count, dtype: int64



Fold results:
Accuracy:      0.843
AUC:           0.793
Recall (Pass): 0.944
F1 (Pass):     0.907
Recall (Fail): 0.413
F1 (Fail):     0.500

Confusion matrix interpretation:
actual_fail_pred_fail = 19 -> correctly identified at-risk students
actual_fail_pred_pass = 27 -> missed at-risk students
actual_pass_pred_fail = 11 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 185 -> correctly identified passing students

GRADIENT BOOSTING - TEST YEAR: 2015
Training size: 911
Test size: 243

Training outcome distribution:
outcome
1    703
0    208
Name: count, dtype: int64

Test outcome distribution:
outcome
1    184
0     59
Name: count, dtype: int64



Fold results:
Accuracy:      0.733
AUC:           0.738
Recall (Pass): 0.902
F1 (Pass):     0.836
Recall (Fail): 0.203
F1 (Fail):     0.270

Confusion matrix interpretation:
actual_fail_pred_fail = 12 -> correctly identified at-risk students
actual_fail_pred_pass = 47 -> missed at-risk students
actual_pass_pred_fail = 18 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 166 -> correctly identified passing students

GRADIENT BOOSTING - TEST YEAR: 2016
Training size: 919
Test size: 235

Training outcome distribution:
outcome
1    713
0    206
Name: count, dtype: int64

Test outcome distribution:
outcome
1    174
0     61
Name: count, dtype: int64



Fold results:
Accuracy:      0.753
AUC:           0.812
Recall (Pass): 0.753
F1 (Pass):     0.819
Recall (Fail): 0.754
F1 (Fail):     0.613

Confusion matrix interpretation:
actual_fail_pred_fail = 46 -> correctly identified at-risk students
actual_fail_pred_pass = 15 -> missed at-risk students
actual_pass_pred_fail = 43 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 131 -> correctly identified passing students

GRADIENT BOOSTING - TEST YEAR: 2017
Training size: 922
Test size: 232

Training outcome distribution:
outcome
1    702
0    220
Name: count, dtype: int64

Test outcome distribution:
outcome
1    185
0     47
Name: count, dtype: int64



Fold results:
Accuracy:      0.823
AUC:           0.833
Recall (Pass): 0.886
F1 (Pass):     0.889
Recall (Fail): 0.574
F1 (Fail):     0.568

Confusion matrix interpretation:
actual_fail_pred_fail = 27 -> correctly identified at-risk students
actual_fail_pred_pass = 20 -> missed at-risk students
actual_pass_pred_fail = 21 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 164 -> correctly identified passing students

GRADIENT BOOSTING - TEST YEAR: 2018
Training size: 952
Test size: 202

Training outcome distribution:
outcome
1    739
0    213
Name: count, dtype: int64

Test outcome distribution:
outcome
1    148
0     54
Name: count, dtype: int64



Fold results:
Accuracy:      0.762
AUC:           0.832
Recall (Pass): 0.953
F1 (Pass):     0.855
Recall (Fail): 0.241
F1 (Fail):     0.351

Confusion matrix interpretation:
actual_fail_pred_fail = 13 -> correctly identified at-risk students
actual_fail_pred_pass = 41 -> missed at-risk students
actual_pass_pred_fail = 7 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 141 -> correctly identified passing students


In [23]:
# ══════════════════════════════════════
# STEP 21: Show Gradient Boosting per-year results
# ══════════════════════════════════════
gb_results_df = pd.DataFrame(gb_results)

print("\n" + "="*60)
print("GRADIENT BOOSTING PER-YEAR CROSS-VALIDATION RESULTS")
print("="*60)
print(gb_results_df.round(3))


GRADIENT BOOSTING PER-YEAR CROSS-VALIDATION RESULTS
   test_year  train_n  test_n  accuracy    auc  recall_pass  f1_pass  \
0       2014      912     242     0.843  0.793        0.944    0.907   
1       2015      911     243     0.733  0.738        0.902    0.836   
2       2016      919     235     0.753  0.812        0.753    0.819   
3       2017      922     232     0.823  0.833        0.886    0.889   
4       2018      952     202     0.762  0.832        0.953    0.855   

   recall_fail  f1_fail  actual_fail_pred_fail  actual_fail_pred_pass  \
0        0.413    0.500                     19                     27   
1        0.203    0.270                     12                     47   
2        0.754    0.613                     46                     15   
3        0.574    0.568                     27                     20   
4        0.241    0.351                     13                     41   

   actual_pass_pred_fail  actual_pass_pred_pass  
0                     11 

# **Overall summary**

Across the five cohort years, the Gradient Boosting model showed **moderate overall performance**, but its usefulness for identifying failing students was uneven. Its ability to distinguish between pass and fail cases was generally acceptable, with **AUC values ranging from 0.738 to 0.833**. However, its success in identifying failing students varied substantially by year. **Fail recall ranged from 0.203 to 0.754**, which indicates that early mastery-test behaviour provided useful signal in some cohorts but was not consistently effective for detecting at-risk students. The strongest year for fail detection was **2016**, while performance was weakest in **2015 and 2018**. 

Overall, Gradient Boosting appeared to favour correct identification of passing students more than failing students. A clear pattern is that Gradient Boosting tended to perform well for the pass class but inconsistently for the fail class. In most years, it achieved high pass recall and respectable accuracy, but this often came at the cost of low sensitivity to failing students.

---
## 2014

In 2014, the model performed fairly well overall, with **accuracy 0.843** and **AUC 0.793**. It was very good at identifying passing students, with **pass recall 0.944**, but less effective at detecting failing students, with **fail recall 0.413**. The model correctly identified only 19 of the 46 failing students and missed 27. This suggests that although the model performed strongly on the majority class, it was relatively weak as an early-warning tool for this cohort.

---

## 2015

This was the weakest year for fail detection. **Accuracy was 0.733** and **AUC was 0.738**, while **fail recall fell to 0.203**, the lowest of all years. The model identified only 12 of the 59 failing students and missed 47. At the same time, it still identified passing students well, with **pass recall 0.902**. This pattern indicates that in 2015 the model was strongly biased toward predicting pass, making it poor at detecting at-risk students.

---

## 2016

The 2016 cohort produced the strongest result for Gradient Boosting in terms of fail detection. The model achieved **AUC 0.812** and **fail recall 0.754**, correctly identifying 46 of the 61 failing students and missing 15. This shows that the model captured meaningful early-risk signal in this cohort. However, it also incorrectly flagged 43 passing students as at risk. Even so, this was the best example of Gradient Boosting working well for the study goal.

---

## 2017

Performance in 2017 was mixed. Overall results were reasonably strong, with **accuracy 0.823** and **AUC 0.833**, and the model identified passing students well, with **pass recall 0.886**. However, **fail recall was only 0.574**, meaning it correctly identified 27 of the 47 failing students and missed 20. This makes 2017 a moderate year for at-risk detection: useful, but clearly less effective than the best-performing cohorts.

---

## 2018

The 2018 cohort again showed the model’s tendency to prioritise pass prediction. **Accuracy was 0.762** and **AUC was 0.832**, while **pass recall reached 0.953**, the highest among the five years. However, **fail recall dropped to 0.241**, meaning the model identified only 13 of the 54 failing students and missed 41. This makes 2018 one of the weakest years for the fail class, despite apparently respectable overall performance measures.

In [24]:
# ══════════════════════════════════════
# STEP 22: Show Gradient Boosting mean and standard deviation
# ══════════════════════════════════════
metric_cols = [
    "accuracy", "auc", "recall_pass",
    "f1_pass", "recall_fail", "f1_fail"
]

gb_summary_df = pd.DataFrame({
    "mean": gb_results_df[metric_cols].mean(),
    "std": gb_results_df[metric_cols].std()
}).round(3)

print("\n" + "="*60)
print("GRADIENT BOOSTING MEAN PERFORMANCE ACROSS ALL YEARS")
print("="*60)
print(gb_summary_df)


GRADIENT BOOSTING MEAN PERFORMANCE ACROSS ALL YEARS
              mean    std
accuracy     0.783  0.048
auc          0.802  0.039
recall_pass  0.888  0.080
f1_pass      0.861  0.036
recall_fail  0.437  0.231
f1_fail      0.461  0.146


# **Gradient Boosting Mean Performance**

### **Accuracy: 0.783 (std 0.048)**
The model correctly predicted about **78% of outcomes**, with fairly steady performance across years.

---

### **AUC: 0.802 (std 0.039)**
Its ability to separate passing and failing students was **good overall** and reasonably consistent.

---

### **Recall (Pass): 0.888 (std 0.080)**
This is very strong.  
The model correctly identified **nearly 89% of passing students**, showing it was highly reliable at recognising students who would succeed.

---

### **F1 (Pass): 0.861 (std 0.036)**
Predictions for passing students were **strong and stable**, with a good balance between precision and recall.

---

### **Recall (Fail): 0.437 (std 0.231)**
This is the weakest area.  
On average, the model identified **only 44% of failing students**, and the large variation shows performance changed a lot from year to year.  
This means the model often **missed many at-risk students**.

---

### **F1 (Fail): 0.461 (std 0.146)**
Performance on failing students was **weak to moderate**, again showing inconsistency and difficulty predicting the at-risk group.

---

# **Overall Summary**
Gradient Boosting was **excellent at identifying passing students**, but **poor and highly inconsistent at detecting failing students**.  
This suggests early mastery-test behaviour strongly signals who will pass, but it is **much less reliable for spotting students at risk of failing**.

In [25]:
# ══════════════════════════════════════
# STEP 23: Save Gradient Boosting outputs
# ══════════════════════════════════════
gb_results_df.to_csv(os.path.join(RESULTS_DIR, "gradient_boosting_loyocv_results.csv"), index=False)
gb_summary_df.to_csv(os.path.join(RESULTS_DIR, "gradient_boosting_loyocv_summary.csv"))

print("\nSaved files:")
print("- gradient_boosting_loyocv_results.csv")
print("- gradient_boosting_loyocv_summary.csv")


Saved files:
- gradient_boosting_loyocv_results.csv
- gradient_boosting_loyocv_summary.csv


In [26]:
# ══════════════════════════════════════
# STEP 24: Support Vector Machine Leave-One-Year-Out CV
# ══════════════════════════════════════
svm_results = []

test_years = sorted(modeling_data["year"].dropna().unique())

for test_year in test_years:
    print("\n" + "="*60)
    print("SUPPORT VECTOR MACHINE - TEST YEAR:", int(test_year))
    print("="*60)

    # Test set = this year
    test_mask = modeling_data["year"] == test_year

    # Training set = all other years
    X_train = X.loc[~test_mask].copy()
    X_test = X.loc[test_mask].copy()
    y_train = y.loc[~test_mask].copy()
    y_test = y.loc[test_mask].copy()

    print("Training size:", X_train.shape[0])
    print("Test size:", X_test.shape[0])

    print("\nTraining outcome distribution:")
    print(y_train.value_counts())

    print("\nTest outcome distribution:")
    print(y_test.value_counts())

    # Scale using training data only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # SVM model
    svm_model = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=42
    )

    svm_model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = svm_model.predict(X_test_scaled)
    y_prob = svm_model.predict_proba(X_test_scaled)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    recall_pass = recall_score(y_test, y_pred, pos_label=1)
    f1_pass = f1_score(y_test, y_pred, pos_label=1)

    recall_fail = recall_score(y_test, y_pred, pos_label=0)
    f1_fail = f1_score(y_test, y_pred, pos_label=0)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    svm_results.append({
        "test_year": int(test_year),
        "train_n": len(X_train),
        "test_n": len(X_test),
        "accuracy": accuracy,
        "auc": auc,
        "recall_pass": recall_pass,
        "f1_pass": f1_pass,
        "recall_fail": recall_fail,
        "f1_fail": f1_fail,
        "actual_fail_pred_fail": tn,
        "actual_fail_pred_pass": fp,
        "actual_pass_pred_fail": fn,
        "actual_pass_pred_pass": tp
    })

    print("\nFold results:")
    print(f"Accuracy:      {accuracy:.3f}")
    print(f"AUC:           {auc:.3f}")
    print(f"Recall (Pass): {recall_pass:.3f}")
    print(f"F1 (Pass):     {f1_pass:.3f}")
    print(f"Recall (Fail): {recall_fail:.3f}")
    print(f"F1 (Fail):     {f1_fail:.3f}")

    print("\nConfusion matrix interpretation:")
    print("actual_fail_pred_fail =", tn, "-> correctly identified at-risk students")
    print("actual_fail_pred_pass =", fp, "-> missed at-risk students")
    print("actual_pass_pred_fail =", fn, "-> students incorrectly flagged as at risk")
    print("actual_pass_pred_pass =", tp, "-> correctly identified passing students")


SUPPORT VECTOR MACHINE - TEST YEAR: 2014
Training size: 912
Test size: 242

Training outcome distribution:
outcome
1    691
0    221
Name: count, dtype: int64

Test outcome distribution:
outcome
1    196
0     46
Name: count, dtype: int64



Fold results:


Accuracy:      0.810
AUC:           0.750
Recall (Pass): 0.872
F1 (Pass):     0.881
Recall (Fail): 0.543
F1 (Fail):     0.521

Confusion matrix interpretation:
actual_fail_pred_fail = 25 -> correctly identified at-risk students
actual_fail_pred_pass = 21 -> missed at-risk students
actual_pass_pred_fail = 25 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 171 -> correctly identified passing students

SUPPORT VECTOR MACHINE - TEST YEAR: 2015
Training size: 911
Test size: 243

Training outcome distribution:
outcome
1    703
0    208
Name: count, dtype: int64

Test outcome distribution:
outcome
1    184
0     59
Name: count, dtype: int64



Fold results:
Accuracy:      0.712
AUC:           0.722
Recall (Pass): 0.734
F1 (Pass):     0.794
Recall (Fail): 0.644
F1 (Fail):     0.521

Confusion matrix interpretation:
actual_fail_pred_fail = 38 -> correctly identified at-risk students
actual_fail_pred_pass = 21 -> missed at-risk students
actual_pass_pred_fail = 49 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 135 -> correctly identified passing students

SUPPORT VECTOR MACHINE - TEST YEAR: 2016
Training size: 919
Test size: 235

Training outcome distribution:
outcome
1    713
0    206
Name: count, dtype: int64

Test outcome distribution:
outcome
1    174
0     61
Name: count, dtype: int64



Fold results:
Accuracy:      0.736
AUC:           0.803
Recall (Pass): 0.713
F1 (Pass):     0.800
Recall (Fail): 0.803
F1 (Fail):     0.613

Confusion matrix interpretation:
actual_fail_pred_fail = 49 -> correctly identified at-risk students
actual_fail_pred_pass = 12 -> missed at-risk students
actual_pass_pred_fail = 50 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 124 -> correctly identified passing students

SUPPORT VECTOR MACHINE - TEST YEAR: 2017
Training size: 922
Test size: 232

Training outcome distribution:
outcome
1    702
0    220
Name: count, dtype: int64

Test outcome distribution:
outcome
1    185
0     47
Name: count, dtype: int64



Fold results:
Accuracy:      0.754
AUC:           0.834
Recall (Pass): 0.746
F1 (Pass):     0.829
Recall (Fail): 0.787
F1 (Fail):     0.565

Confusion matrix interpretation:
actual_fail_pred_fail = 37 -> correctly identified at-risk students
actual_fail_pred_pass = 10 -> missed at-risk students
actual_pass_pred_fail = 47 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 138 -> correctly identified passing students

SUPPORT VECTOR MACHINE - TEST YEAR: 2018
Training size: 952
Test size: 202

Training outcome distribution:
outcome
1    739
0    213
Name: count, dtype: int64

Test outcome distribution:
outcome
1    148
0     54
Name: count, dtype: int64



Fold results:
Accuracy:      0.787
AUC:           0.783
Recall (Pass): 0.919
F1 (Pass):     0.863
Recall (Fail): 0.426
F1 (Fail):     0.517

Confusion matrix interpretation:
actual_fail_pred_fail = 23 -> correctly identified at-risk students
actual_fail_pred_pass = 31 -> missed at-risk students
actual_pass_pred_fail = 12 -> students incorrectly flagged as at risk
actual_pass_pred_pass = 136 -> correctly identified passing students


In [27]:
# ══════════════════════════════════════
# STEP 25: Show SVM per-year results
# ══════════════════════════════════════
svm_results_df = pd.DataFrame(svm_results)

print("\n" + "="*60)
print("SVM PER-YEAR CROSS-VALIDATION RESULTS")
print("="*60)
print(svm_results_df.round(3))


SVM PER-YEAR CROSS-VALIDATION RESULTS
   test_year  train_n  test_n  accuracy    auc  recall_pass  f1_pass  \
0       2014      912     242     0.810  0.750        0.872    0.881   
1       2015      911     243     0.712  0.722        0.734    0.794   
2       2016      919     235     0.736  0.803        0.713    0.800   
3       2017      922     232     0.754  0.834        0.746    0.829   
4       2018      952     202     0.787  0.783        0.919    0.863   

   recall_fail  f1_fail  actual_fail_pred_fail  actual_fail_pred_pass  \
0        0.543    0.521                     25                     21   
1        0.644    0.521                     38                     21   
2        0.803    0.612                     49                     12   
3        0.787    0.565                     37                     10   
4        0.426    0.517                     23                     31   

   actual_pass_pred_fail  actual_pass_pred_pass  
0                     25               

# **Overall Summary**

Across the five cohort years, the Support Vector Machine showed **moderate predictive performance overall**. Its ability to distinguish between passing and failing students was generally acceptable, with **AUC values ranging from 0.722 to 0.834**. However, its effectiveness in identifying failing students varied across years. **Fail recall ranged from 0.426 to 0.803**, indicating that early mastery-test behaviour provided useful predictive signal, but not with equal strength in every cohort. The strongest years for detecting at-risk students were **2016 and 2017**, while performance was weaker in **2014 and 2018**, and more mixed in **2015**.  

A clear pattern is that the SVM, like Logistic Regression and Random Forest, performed best for identifying at-risk students in **2016 and 2017**, and more weakly in **2014 and 2018**. In the stronger years, the model achieved fairly high fail recall, but this often came with a substantial number of false positives. Overall, the SVM appears to offer useful early-risk detection, but like the other models, its sensitivity to failing students is cohort-dependent.

---
## 2014

In 2014, the SVM performed reasonably well overall, with **accuracy 0.810** and **AUC 0.750**. It identified passing students fairly well, with **pass recall 0.872**, but was only moderately effective at detecting failing students, with **fail recall 0.543**. The model correctly identified 25 of the 46 failing students and missed 21. This suggests that the early mastery features contained useful information, but the model was not strong enough to provide highly reliable at-risk detection in this cohort.

---

## 2015

This was one of the weaker years overall, although fail detection was somewhat better than in 2014. **Accuracy was 0.712** and **AUC was 0.722**, the lowest AUC across the five cohorts. **Fail recall reached 0.644**, meaning the model correctly identified 38 of the 59 failing students and missed 21. However, it also incorrectly flagged 49 passing students as at risk. This indicates that the model was more sensitive to failing students in 2015, but at the cost of many false positives, reducing overall reliability.

---
## 2016

The 2016 cohort produced one of the strongest results for SVM. The model achieved **AUC 0.803** and a high **fail recall of 0.803**, correctly identifying 49 of the 61 failing students and missing only 12. This is a strong result for your research aim. However, the model also incorrectly flagged 50 passing students as at risk. Even so, the 2016 fold shows that early mastery-level behaviour can be quite informative for fail detection when used with this classifier.

---

## 2017

This was also a strong year for SVM. The model achieved the highest **AUC (0.834)** and a high **fail recall (0.787)**. It correctly identified 37 of the 47 failing students and missed only 10. Although it produced 47 false positives, the model was clearly sensitive to at-risk students in this cohort. Overall, 2017 provides good evidence that the Week 1-3 mastery features can support useful fail prediction.

---

## 2018

The 2018 cohort showed a different pattern. Overall performance remained fairly solid, with **accuracy 0.787** and **AUC 0.783**, while **pass recall was very high at 0.919**. However, **fail recall dropped to 0.426**, meaning the model correctly identified only 23 of the 54 failing students and missed 31. This indicates that, in 2018, the model was much better at recognising students who would pass than those who were at risk of failure.

In [28]:
# ══════════════════════════════════════
# STEP 26: Show SVM mean and standard deviation
# ══════════════════════════════════════
metric_cols = [
    "accuracy", "auc", "recall_pass",
    "f1_pass", "recall_fail", "f1_fail"
]

svm_summary_df = pd.DataFrame({
    "mean": svm_results_df[metric_cols].mean(),
    "std": svm_results_df[metric_cols].std()
}).round(3)

print("\n" + "="*60)
print("SVM MEAN PERFORMANCE ACROSS ALL YEARS")
print("="*60)
print(svm_summary_df)


SVM MEAN PERFORMANCE ACROSS ALL YEARS
              mean    std
accuracy     0.760  0.039
auc          0.779  0.044
recall_pass  0.797  0.093
f1_pass      0.834  0.038
recall_fail  0.641  0.161
f1_fail      0.547  0.042


# **SVM Mean Performance**

### **Accuracy: 0.760 (std 0.039)**
The model correctly predicted about **76% of outcomes**, with fairly stable performance across years.

---

### **AUC: 0.779 (std 0.044)**
Its ability to distinguish passing from failing students was **good but slightly weaker** than the other models.  
Still, the variation across years was small, showing consistent behaviour.

---

### **Recall (Pass): 0.797 (std 0.093)**
The model correctly identified about **80% of passing students**, with some year-to-year fluctuation.

---

### **F1 (Pass): 0.834 (std 0.038)**
Predictions for passing students were **strong and reliable**, with a good balance between precision and recall.

---

### **Recall (Fail): 0.641 (std 0.161)**
On average, the model identified **64% of failing students**, which is moderate.  
The large standard deviation shows that performance for the at-risk group **varied a lot between cohorts**.

---

### **F1 (Fail): 0.547 (std 0.042)**
Performance on failing students was **moderate**, indicating the model struggled more with this group than with passing students.

---

# **Overall Summary**
SVM performed solidly overall and was consistently good at identifying passing students. Its ability to detect failing students was **moderate but unstable**, suggesting that early mastery-test behaviour was useful but not equally predictive across all cohorts.

In [29]:
# ══════════════════════════════════════
# STEP 27: Save SVM outputs
# ══════════════════════════════════════
svm_results_df.to_csv(os.path.join(RESULTS_DIR, "svm_loyocv_results.csv"), index=False)
svm_summary_df.to_csv(os.path.join(RESULTS_DIR, "svm_loyocv_summary.csv"))

print("\nSaved files:")
print("- svm_loyocv_results.csv")
print("- svm_loyocv_summary.csv")


Saved files:
- svm_loyocv_results.csv
- svm_loyocv_summary.csv


In [30]:
# ══════════════════════════════════════
# STEP 28: Compare model mean performance
# ══════════════════════════════════════
comparison_df = pd.DataFrame({
    "Logistic Regression": summary_df["mean"],
    "Random Forest": rf_summary_df["mean"],
    "Gradient Boosting": gb_summary_df["mean"],
    "SVM": svm_summary_df["mean"]
}).round(3)

print("\n" + "="*60)
print("MODEL COMPARISON: MEAN PERFORMANCE")
print("="*60)
print(comparison_df)


MODEL COMPARISON: MEAN PERFORMANCE
             Logistic Regression  Random Forest  Gradient Boosting    SVM
accuracy                   0.766          0.757              0.783  0.760
auc                        0.803          0.784              0.802  0.779
recall_pass                0.800          0.798              0.888  0.797
f1_pass                    0.837          0.831              0.861  0.834
recall_fail                0.661          0.625              0.437  0.641
f1_fail                    0.561          0.537              0.461  0.547


# **Model Comparison Mean Performance**

## **Overall Accuracy**
- All models sit around **75-78% accuracy**.
- **Gradient Boosting** is the highest (0.783), but only slightly above the others.
- Accuracy differences are small, so it’s not the best metric for choosing between models.

---

## **AUC (ability to separate pass vs fail)**
- **Logistic Regression (0.803)** and **Gradient Boosting (0.802)** perform the best.
- **Random Forest (0.784)** and **SVM (0.779)** are slightly weaker.
- All models show **good but not perfect** discrimination.

---

## **Performance on Passing Students**
- All models do well here.
- **Gradient Boosting** is strongest with **recall_pass 0.888** and **f1_pass 0.861**.
- Logistic Regression, Random Forest, and SVM are all very similar (around 0.80 recall, 0.83 F1).
- This shows that **predicting who will pass is easy for all models**.

---

## **Performance on Failing Students (the important part)**
This is where the models differ the most.

### **Fail Recall (how many at-risk students are caught)**
- **Logistic Regression: 0.661** → best overall
- **SVM: 0.641** → close behind
- **Random Forest: 0.625** → moderate
- **Gradient Boosting: 0.437** → weakest by far

### **Fail F1 (balance of precision + recall)**
- **Logistic Regression: 0.561** → strongest
- **SVM: 0.547** → similar
- **Random Forest: 0.537** → slightly weaker
- **Gradient Boosting: 0.461** → lowest

**Conclusion:**  
Logistic Regression and SVM are the most reliable for detecting failing students.  
Gradient Boosting is excellent for predicting passes but **poor at identifying at-risk students**.

---

# **Overall Summary**
- **Gradient Boosting** is the best at predicting who will pass, but the worst at detecting who will fail.
- **Logistic Regression** offers the best balance, with the strongest and most stable performance for the fail class.
- **SVM** is similar to Logistic Regression but slightly weaker.
- **Random Forest** performs reasonably well but is less consistent for failing students.

Conclusill models can predict passing students well, but **Logistic Regression and SVM are the most effective for identifying at-risk students**, which is the key goal of an early-warning system.
